# Bidless Dataset Diagnostics

Comprehensive diagnostic toolkit for analyzing bidless hand datasets used in ML model training.

## What are Bidless Hands?

**Bidless hands** are 10-card hands captured **after bidding has concluded** (contract and trump already declared). These datasets enable training ML models for hand evaluation without simulating the bidding phase.

**Key components:**
- Hand cards (5 cards per player, 2 players per team)
- Contract type (suit/high/low)
- Trump suit (for suit contracts)
- Hand features (31 from bidless schema or 40+ from hand_eval)
- Optionally: actual outcomes (tricks_won from simulation)

**Use case:** Train models to predict hand value/strength given a declared contract, enabling better bidding decisions.

## Two Chart Families

This notebook demonstrates two chart families:

| Module | Input Type | Use Case | Display Method |
|--------|------------|----------|----------------|
| `bid_euchre.diagnostics` | **DataFrame** | Interactive dataset analysis | `plt.show()` |
| `bid_euchre.reporting.validation` | **Dict/List** | Batch report generation | Saves to disk |

**When to use each:**
- **Diagnostics**: Exploratory analysis in notebooks, quick health checks
- **Reporting/Eval**: Training pipelines, reproducible batch reports

## Two Data Types

1. **Features-only**: Hand features without simulation (fast, used for feature health checks)
2. **Features + Outcomes**: Includes `tricks_won` from simulation (required for predictive accuracy analysis)

## Notebook Structure

- **Part 1**: Data generation (features + outcomes)
- **Part 2**: Core bidless-specific analysis (NEW)
- **Part 3**: General feature diagnostics
- **Part 4**: Dict-based evaluation charts
- **Part 5**: Additional outcome analysis
- **Part 6**: Strategy performance on bidless hands
- **Part 7**: Trump suit analysis
- **Part 8**: Distribution analysis (CDF/CCDF)

In [ ]:
# Auto-reload for development
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import shutil
import tempfile

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image

from bid_euchre.diagnostics import (
    plot_feature_correlation,
    plot_feature_distributions,
    plot_hand_value_by_contract,
    plot_hand_value_by_seat,
    plot_rolling_mean,
)
from bid_euchre.diagnostics.charts import plot_feature_vs_label
from bid_euchre.features.hand_eval import get_hand_features
from bid_euchre.reporting.validation import (
    generate_validation_plots,
)
from bid_euchre.reporting.validation import (
    plot_feature_correlation as eval_plot_feature_correlation,
)
from bid_euchre.reporting.validation import (
    plot_feature_distributions as eval_plot_feature_distributions,
)
from bid_euchre.reporting.validation import (
    plot_hand_value_by_contract as eval_plot_hand_value_by_contract,
)
from bid_euchre.sim.deals import generate_deal

plt.style.use("seaborn-v0_8-whitegrid")

---
# Part 1: Data Generation

Generate sample datasets for demonstration, including both features and outcomes.

In [ ]:
# Generate DataFrame for diagnostic charts
SEED = 42
N_DEALS = 200

hands_data = []
for deal_id in range(N_DEALS):
    hands = generate_deal(SEED, deal_id)
    # Vary contract types
    contract_types = ['suit', 'high', 'low']
    contract_type = contract_types[deal_id % 3]
    trump = 'H' if contract_type == 'suit' else None
    
    for seat in range(4):
        hand = hands[seat]
        features = get_hand_features(hand, contract_type, trump)
        hands_data.append({
            'hand_id': f"{deal_id}_{seat}",
            'deal_id': deal_id,
            'seat': seat,
            'contract_type': contract_type,
            'trump': trump,
            # Add feat_ prefix for diagnostic charts
            **{f'feat_{k}': v for k, v in features.items()}
        })

df = pd.DataFrame(hands_data)
print(f"DataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)[:10]}...")
df.head(3)

In [ ]:
# Generate Dict structure for evaluation charts
features_by_contract = {'suit_H': [], 'high': [], 'low': []}

for deal_id in range(N_DEALS):
    hands = generate_deal(SEED, deal_id)
    
    for contract_key, contract_type, trump in [
        ('suit_H', 'suit', 'H'),
        ('high', 'high', None),
        ('low', 'low', None),
    ]:
        # Just use seat 0 for simplicity
        features = get_hand_features(hands[0], contract_type, trump)
        features_by_contract[contract_key].append(features)

print(f"Dict structure: {list(features_by_contract.keys())}")
print(f"Samples per contract: {len(features_by_contract['suit_H'])}")

## 1.1 Generate Outcome Data (features + tricks_won)

Run simulations to create a dataset with both hand features and actual trick outcomes.
This data is required for Parts 2 and 5.

In [ ]:
# Generate simulation data with features + tricks_won
from bid_euchre.sim.simulation import play_single_hand
from bid_euchre.strategy import GreedyStrategy

# Run simulations to get outcome data
N_DEALS_SIM = 200
outcome_data = []
strategy = GreedyStrategy()

for deal_id in range(N_DEALS_SIM):
    hands = generate_deal(SEED, deal_id)
    
    for contract_type in ['suit', 'high', 'low']:
        trump = 'H' if contract_type == 'suit' else None
        
        t0, t1, _, all_feats, _, _, *_ = play_single_hand(
            contract_type=contract_type,
            trump_suit=trump,
            strategy=strategy,
            hands=hands,
            deal_seed=SEED,
        )
        
        # Record each player's features + their team's tricks
        for seat in range(4):
            team_tricks = t0 if seat in (0, 2) else t1
            features = all_feats[seat]
            outcome_data.append({
                'deal_id': deal_id,
                'seat': seat,
                'contract_type': contract_type,
                'trump': trump,
                'tricks_won': team_tricks,
                **{f'feat_{k}': v for k, v in features.items()}
            })

outcome_df = pd.DataFrame(outcome_data)
print(f"Simulation DataFrame shape: {outcome_df.shape}")
print(f"Tricks won range: {outcome_df['tricks_won'].min()} - {outcome_df['tricks_won'].max()}")
outcome_df.head(3)

---
# Part 2: Core Bidless Analysis

NEW: Charts specifically designed for bidless dataset quality and predictive power analysis.

## 2.1 hand_value vs tricks_won - Predictive Accuracy

**Purpose**: Evaluate how well estimated hand strength (hand_value) predicts actual outcomes (tricks_won).

**Key insight**: High R² indicates hand_value is a reliable proxy for hand strength. Large residuals identify where the model fails.

In [ ]:
import numpy as np
from scipy import stats

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter with regression line
ax1.scatter(outcome_df['feat_hand_value'], outcome_df['tricks_won'], alpha=0.3, s=10)
slope, intercept, r_value, p_value, std_err = stats.linregress(
    outcome_df['feat_hand_value'], outcome_df['tricks_won']
)
line_x = np.array([outcome_df['feat_hand_value'].min(), outcome_df['feat_hand_value'].max()])
line_y = slope * line_x + intercept
ax1.plot(line_x, line_y, 'r-', linewidth=2, label=f'R² = {r_value**2:.3f}')
ax1.set_xlabel('hand_value (estimated strength)', fontsize=11)
ax1.set_ylabel('tricks_won (actual outcome)', fontsize=11)
ax1.set_title('Predictive Accuracy: hand_value vs tricks_won', fontsize=12, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Right: residual plot
residuals = outcome_df['tricks_won'] - (slope * outcome_df['feat_hand_value'] + intercept)
ax2.scatter(outcome_df['feat_hand_value'], residuals, alpha=0.3, s=10)
ax2.axhline(y=0, color='r', linestyle='--', linewidth=2)
ax2.set_xlabel('hand_value (estimated)', fontsize=11)
ax2.set_ylabel('Residuals (actual - predicted)', fontsize=11)
ax2.set_title('Prediction Errors', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Correlation: {r_value:.3f}")
print(f"R²: {r_value**2:.3f} (proportion of variance explained)")
print(f"RMSE: {np.sqrt(np.mean(residuals**2)):.3f} tricks")
print("\nInterpretation:")
if r_value**2 > 0.5:
    print("  ✓ Strong predictive power - hand_value reliably estimates outcomes")
else:
    print("  ⚠ Weak predictive power - hand_value may need improvement")

## 2.2 Feature Stability Across Contracts

**Purpose**: Verify features behave consistently across suit/high/low contract types.

**Key insight**: Stable features (similar distributions) generalize well. Highly variable features may need contract-specific handling.

In [ ]:
# Select key features to analyze
key_features = ['hand_value', 'trump_count', 'offsuit_aces', 'offsuit_kings']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, feature in enumerate(key_features):
    ax = axes[idx]

    for contract in ['suit', 'high', 'low']:
        contract_df = df[df['contract_type'] == contract]
        col = f'feat_{feature}'
        if col in contract_df.columns:
            ax.hist(contract_df[col], alpha=0.5, label=contract, bins=30, edgecolor='black', linewidth=0.5)

    ax.set_xlabel(feature, fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'{feature} - Distribution by Contract', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Feature Stability Across Contract Types', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

# Calculate variance ratios to identify unstable features
print("Feature Variance by Contract Type:")
print("=" * 60)
for feature in key_features:
    col = f'feat_{feature}'
    if col in df.columns:
        variances = df.groupby('contract_type')[col].var()
        mean_var = variances.mean()
        max_ratio = variances.max() / variances.min() if variances.min() > 0 else float('inf')

        print(f"\n{feature}:")
        print(variances.to_string())
        print(f"  Max/Min ratio: {max_ratio:.2f}x")

        if max_ratio > 3:
            print("  ⚠ HIGH VARIANCE - may need contract-specific handling")
        else:
            print("  ✓ Stable across contracts")

## 2.3 Contract-Specific Feature Importance

**Purpose**: Identify which features matter most for each contract type (suit/high/low).

**Key insight**: Different contracts may rely on different features. Common important features across contracts should be prioritized in model training.

In [ ]:
# Calculate feature importance (correlation with tricks_won) for each contract
feat_cols = [col for col in outcome_df.columns if col.startswith('feat_')]
top_n = 10

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
contracts = ['suit', 'high', 'low']

# Store top features for each contract
contract_top_features = {}

for idx, contract in enumerate(contracts):
    contract_df = outcome_df[outcome_df['contract_type'] == contract]

    # Calculate correlations
    correlations = {}
    for col in feat_cols:
        corr = contract_df[col].corr(contract_df['tricks_won'])
        correlations[col.replace('feat_', '')] = corr

    # Get top N by absolute value
    top_feats = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)[:top_n]
    contract_top_features[contract] = set([f[0] for f in top_feats])

    # Plot
    ax = axes[idx]
    features = [f[0] for f in top_feats]
    values = [f[1] for f in top_feats]
    colors = ['green' if v > 0 else 'red' for v in values]

    ax.barh(features, values, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.set_xlabel('Correlation with tricks_won', fontsize=11)
    ax.set_title(f'{contract.upper()} Contract', fontsize=12, fontweight='bold')
    ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()

plt.suptitle('Contract-Specific Feature Importance (Top 10 by |correlation|)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Analyze feature overlap
suit_feats = contract_top_features['suit']
high_feats = contract_top_features['high']
low_feats = contract_top_features['low']

print("\nFeature Overlap Analysis:")
print("=" * 60)
print(f"All three contracts: {sorted(suit_feats & high_feats & low_feats)}")
print(f"Suit + High only: {sorted(suit_feats & high_feats - low_feats)}")
print(f"Suit + Low only: {sorted(suit_feats & low_feats - high_feats)}")
print(f"High + Low only: {sorted(high_feats & low_feats - suit_feats)}")
print(f"Suit only: {sorted(suit_feats - high_feats - low_feats)}")
print(f"High only: {sorted(high_feats - suit_feats - low_feats)}")
print(f"Low only: {sorted(low_feats - suit_feats - high_feats)}")

common_count = len(suit_feats & high_feats & low_feats)
print(f"\n✓ {common_count}/{top_n} features are important across all contracts")

## 2.4 Seat Position Effects

**Purpose**: Analyze whether dealer/leader position or seat assignment impacts hand strength or outcomes.

**Key insight**: Significant differences indicate:
- Dealing bias (seat-to-seat variations)
- Positional advantages (dealer/leader effects)

Bidless features include dealer/leader one-hot encodings, so understanding their impact is critical for model training.

In [ ]:
from scipy.stats import f_oneway

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. hand_value by dealer position
ax1 = axes[0, 0]
dealer_data = [df[df['seat'] == i]['feat_hand_value'].values for i in range(4)]
bp1 = ax1.boxplot(dealer_data, labels=['Seat 0', 'Seat 1', 'Seat 2', 'Seat 3'], patch_artist=True)
for patch in bp1['boxes']:
    patch.set_facecolor('lightblue')
ax1.set_ylabel('hand_value', fontsize=11)
ax1.set_title('Hand Value by Dealer Position', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# 2. tricks_won by dealer position (from outcome_df)
ax2 = axes[0, 1]
dealer_outcome = [outcome_df[outcome_df['seat'] == i]['tricks_won'].values for i in range(4)]
bp2 = ax2.boxplot(dealer_outcome, labels=['Seat 0', 'Seat 1', 'Seat 2', 'Seat 3'], patch_artist=True)
for patch in bp2['boxes']:
    patch.set_facecolor('lightgreen')
ax2.set_ylabel('tricks_won', fontsize=11)
ax2.set_title('Tricks Won by Dealer Position', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# 3. hand_value by seat (detect dealing bias)
ax3 = axes[1, 0]
seat_data = [df[df['seat'] == i]['feat_hand_value'].values for i in range(4)]
bp3 = ax3.boxplot(seat_data, labels=['Seat 0', 'Seat 1', 'Seat 2', 'Seat 3'], patch_artist=True)
for patch in bp3['boxes']:
    patch.set_facecolor('lightyellow')
ax3.set_ylabel('hand_value', fontsize=11)
ax3.set_title('Hand Value by Seat (Dealing Bias Check)', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

# 4. Statistical summary
ax4 = axes[1, 1]
ax4.axis('off')

# ANOVA tests
f_stat1, p_value1 = f_oneway(*dealer_data)
f_stat2, p_value2 = f_oneway(*dealer_outcome)
f_stat3, p_value3 = f_oneway(*seat_data)

summary_text = "Statistical Tests (ANOVA):\n"
summary_text += "=" * 50 + "\n\n"

summary_text += "Dealer position → hand_value\n"
summary_text += f"  F-statistic: {f_stat1:.3f}\n"
summary_text += f"  p-value: {p_value1:.4f}\n"
summary_text += f"  Significant: {'Yes (⚠)' if p_value1 < 0.05 else 'No (✓)'}\n\n"

summary_text += "Dealer position → tricks_won\n"
summary_text += f"  F-statistic: {f_stat2:.3f}\n"
summary_text += f"  p-value: {p_value2:.4f}\n"
summary_text += f"  Significant: {'Yes (⚠)' if p_value2 < 0.05 else 'No (✓)'}\n\n"

summary_text += "Seat dealing bias check\n"
summary_text += f"  F-statistic: {f_stat3:.3f}\n"
summary_text += f"  p-value: {p_value3:.4f}\n"
summary_text += f"  Bias detected: {'Yes (⚠)' if p_value3 < 0.05 else 'No (✓)'}\n\n"

summary_text += "\nInterpretation:\n"
summary_text += "  p < 0.05: Position significantly affects outcome\n"
summary_text += "  p ≥ 0.05: No significant position effect\n\n"

if p_value3 < 0.05:
    summary_text += "⚠ Dealing bias detected - check RNG!\n"
else:
    summary_text += "✓ No dealing bias - RNG is fair\n"

ax4.text(0.05, 0.5, summary_text, fontsize=10, family='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.suptitle('Seat Position Effects on Hand Strength and Outcomes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Part 3: Feature Distribution Analysis

From `bid_euchre.diagnostics` — interactive DataFrame-based analysis for exploratory work.

**When to use**: Jupyter notebooks, quick health checks, interactive exploration
**Input format**: pandas DataFrame with `feat_*` columns
**Display**: `plt.show()` (inline in notebook)

## 1.1 `plot_hand_value_by_seat()`

Box plots showing hand_value distribution across seats (0-3). Detects dealing bias or per-seat feature computation bugs.

In [ ]:
fig = plot_hand_value_by_seat(df)
plt.show()

## 1.2 `plot_hand_value_by_contract()`

Box plots comparing hand_value across contract types (suit/high/low).

In [ ]:
fig = plot_hand_value_by_contract(df)
plt.show()

## 1.3 `plot_feature_distributions()`

Grid of histograms for multiple features. Shows top 9 features by variance by default.

In [ ]:
fig = plot_feature_distributions(df)
plt.show()

In [ ]:
# With specific features
fig = plot_feature_distributions(df, features=['hand_value', 'trump_count', 'offsuit_aces'])
plt.show()

## 1.4 `plot_feature_correlation()`

Heatmap of feature correlations. Top 10 features by variance by default.

In [ ]:
fig = plot_feature_correlation(df)
plt.show()

## 1.5 `plot_rolling_mean()`

Time-series plot of rolling mean over hand index. Detects drift over time.

In [ ]:
fig = plot_rolling_mean(df, column='feat_hand_value', window=50)
plt.show()

## 1.6 `plot_feature_vs_label()`

Dual panel: scatter plot + binned box plot. Shows feature vs label relationship.

In [ ]:
fig = plot_feature_vs_label(df, feature='trump_count', label='hand_value')
plt.show()

---
# Part 4: Evaluation Charts (Dict-based, for Reporting Pipelines)

From `bid_euchre.reporting.validation` — batch report generation for training pipelines.

**When to use**: Reproducible batch reports, training pipeline integration
**Input format**: `Dict[contract_key, List[feature_dict]]` or `List[feature_dict]`
**Output**: Saves PNG files to disk, returns file paths

In [ ]:
# Create temp directory for output
output_dir = tempfile.mkdtemp(prefix='charts_demo_')
print(f"Output directory: {output_dir}")

## 2.1 `plot_feature_distributions()` (eval)

Feature distributions by contract type. Overlays histograms for comparison.

In [ ]:
path = eval_plot_feature_distributions(
    features_by_contract,
    output_dir,
    feature_keys=["trump_count", "hand_value"],
)
print(f"Saved to: {path}")

# Display the saved image
Image(filename=path)

## 2.2 `plot_feature_correlation()` (eval)

Correlation matrix from feature dicts. Auto-detects numeric columns.

In [ ]:
# Flatten all features for correlation
all_features = []
for features_list in features_by_contract.values():
    all_features.extend(features_list)

path = eval_plot_feature_correlation(all_features, output_dir)
print(f"Saved to: {path}")

Image(filename=path)

## 2.3 `plot_hand_value_by_contract()` (eval)

Box plots of hand_value by contract type from Dict input.

In [ ]:
path = eval_plot_hand_value_by_contract(features_by_contract, output_dir)
print(f"Saved to: {path}")

Image(filename=path)

## 2.4 `generate_validation_plots()`

Orchestrator function that generates all evaluation plots at once.

In [ ]:
# Use a separate subdirectory
batch_dir = os.path.join(output_dir, "batch")

plots = generate_validation_plots(features_by_contract, batch_dir)
print("Generated plots:")
for name, path in plots.items():
    print(f"  {name}: {path}")

---
# Part 5: Additional Outcome Evaluation

Additional charts that correlate hand features with actual outcomes (`tricks_won`) from simulation.

**Key requirement**: Uses `outcome_df` generated in Part 1.

## 3.1 `plot_feature_vs_outcome()` - hand_value vs tricks_won

Scatter plot with trend line + binned box plot. Shows correlation coefficient in title.

In [ ]:
from bid_euchre.diagnostics.charts import plot_feature_vs_outcome

# hand_value vs tricks_won - the key relationship
fig = plot_feature_vs_outcome(outcome_df, feature='hand_value', outcome='tricks_won')
plt.show()

In [ ]:
# Also check trump_count vs tricks_won for suit contracts
suit_df = outcome_df[outcome_df['contract_type'] == 'suit']
fig = plot_feature_vs_outcome(suit_df, feature='trump_count', outcome='tricks_won')
plt.show()

## 3.2 `plot_outcome_distributions()` - tricks by contract type

Violin/box plots of outcome distribution grouped by category.

In [ ]:
from bid_euchre.diagnostics.charts import plot_outcome_distributions

fig = plot_outcome_distributions(outcome_df, outcome='tricks_won', group_by='contract_type')
plt.show()

## 3.3 `plot_feature_outcome_correlation()` - feature importance bar chart

Horizontal bar chart showing correlation of each feature with tricks_won, sorted by importance.

In [ ]:
from bid_euchre.diagnostics.charts import plot_feature_outcome_correlation

fig = plot_feature_outcome_correlation(outcome_df, outcome='tricks_won', top_n=15)
plt.show()

In [ ]:
# Filter to suit contracts only for more specific feature importance
fig = plot_feature_outcome_correlation(suit_df, outcome='tricks_won', top_n=15)
plt.show()

---
# Part 6: Strategy Performance on Bidless Hands

Test play strategy effectiveness using declared contracts (bidless data, no auction).

**Scope**: Compares play strategies (Greedy, Glutton, Random) on hands with pre-declared contracts. This is bidless analysis because contracts are randomly assigned, not determined by bidding.

**Use case**:
- Validate strategy implementations
- Identify which strategies perform best given declared contracts
- Sanity check that simulations are fair (self-play should average 5 tricks)

## 4.0 Generate Matchup Data

Run simulations with different strategy pairs to collect matchup results.

In [ ]:
import numpy as np

from bid_euchre.strategy import GluttonStrategy, RandomLegalStrategy

# Define strategies to compare
STRATEGY_CLASSES = {
    'greedy': GreedyStrategy,
    'glutton': GluttonStrategy,
    'random': lambda: RandomLegalStrategy(seed=42),
}

# Generate matchup results
N_HANDS = 200
matchup_results = {}

for team0_name in STRATEGY_CLASSES.keys():
    for team1_name in STRATEGY_CLASSES.keys():
        team0_strat = STRATEGY_CLASSES[team0_name]()
        team1_strat = STRATEGY_CLASSES[team1_name]()
        strategies = [team0_strat, team1_strat, team0_strat, team1_strat]
        
        t0_tricks = []
        t1_tricks = []
        
        for deal_id in range(N_HANDS):
            hands = generate_deal(SEED, deal_id)
            t0, t1, *_ = play_single_hand(
                contract_type='suit',
                trump_suit='H',
                strategies=strategies,
                hands=hands,
                deal_seed=SEED,
            )
            t0_tricks.append(t0)
            t1_tricks.append(t1)
        
        matchup_results[(team0_name, team1_name)] = {
            'tricks_team0': t0_tricks,
            'tricks_team1': t1_tricks,
            'mean_tricks': np.mean(t0_tricks),
            'win_rate': np.mean([1 if t >= 6 else 0 for t in t0_tricks]),
            'ci_lower': np.mean(t0_tricks) - 1.96 * np.std(t0_tricks) / np.sqrt(N_HANDS),
            'ci_upper': np.mean(t0_tricks) + 1.96 * np.std(t0_tricks) / np.sqrt(N_HANDS),
        }

print(f"Generated {len(matchup_results)} matchups")
for key, result in matchup_results.items():
    print(f"  {key[0]} vs {key[1]}: mean={result['mean_tricks']:.2f}, win_rate={result['win_rate']:.1%}")

## 4.1 `plot_win_rate_heatmap()` - all-vs-all win rates

Heatmap showing Team 0's win rate against each Team 1 strategy.

In [ ]:
from bid_euchre.diagnostics import plot_win_rate_heatmap

fig = plot_win_rate_heatmap(matchup_results, metric='win_rate')
plt.show()

## 4.2 `plot_tricks_distribution_comparison()` - violin plots by matchup

Violin plots comparing trick distributions across different matchups.

In [ ]:
from bid_euchre.diagnostics import plot_tricks_distribution_comparison

# Show a subset of interesting matchups
subset_matchups = {k: v for k, v in matchup_results.items() 
                   if k[0] != k[1]}  # Exclude self-play for comparison

fig = plot_tricks_distribution_comparison(subset_matchups, team=0)
plt.show()

## 4.3 `plot_strategy_delta_bars()` - delta vs baseline

Bar chart showing mean tricks delta relative to baseline (random).

In [ ]:
from bid_euchre.diagnostics import plot_strategy_delta_bars

# Compare each strategy vs random baseline
baseline_results = matchup_results[('random', 'random')]
comparison_results = {
    'greedy': matchup_results[('greedy', 'random')],
    'glutton': matchup_results[('glutton', 'random')],
}

fig = plot_strategy_delta_bars(baseline_results, comparison_results, baseline_name='random')
plt.show()

## 4.4 `plot_self_play_control()` - self-play sanity check

Control chart showing mean tricks for self-play matchups. Should be ~5.0 for fair play.

In [ ]:
from bid_euchre.diagnostics import plot_self_play_control

# Extract self-play matchups
self_play_results = {k[0]: v for k, v in matchup_results.items() if k[0] == k[1]}

fig = plot_self_play_control(self_play_results)
plt.show()

---
# Part 7: Trump Suit Analysis

Charts comparing distributions and outcomes across trump suits (C, D, H, S).

**Key question**: Do certain trump suits lead to systematically different hand strengths or outcomes?

## 5.0 Generate Multi-Suit Data

Generate data with all 4 trump suits for comparison.

In [ ]:
# Generate data with all 4 trump suits
multi_suit_data = []
multi_suit_outcome_data = []

for deal_id in range(N_DEALS_SIM):
    hands = generate_deal(SEED, deal_id)
    
    for trump in ['C', 'D', 'H', 'S']:
        # Features only (no simulation)
        for seat in range(4):
            features = get_hand_features(hands[seat], 'suit', trump)
            multi_suit_data.append({
                'deal_id': deal_id,
                'seat': seat,
                'contract_type': 'suit',
                'trump': trump,
                **{f'feat_{k}': v for k, v in features.items()}
            })
        
        # With simulation outcomes
        t0, t1, _, all_feats, _, _, *_ = play_single_hand(
            contract_type='suit',
            trump_suit=trump,
            strategy=strategy,
            hands=hands,
            deal_seed=SEED,
        )
        
        for seat in range(4):
            team_tricks = t0 if seat in (0, 2) else t1
            features = all_feats[seat]
            multi_suit_outcome_data.append({
                'deal_id': deal_id,
                'seat': seat,
                'contract_type': 'suit',
                'trump': trump,
                'tricks_won': team_tricks,
                **{f'feat_{k}': v for k, v in features.items()}
            })

multi_suit_df = pd.DataFrame(multi_suit_data)
multi_suit_outcome_df = pd.DataFrame(multi_suit_outcome_data)

print(f"Multi-suit DataFrame: {multi_suit_df.shape}")
print(f"Trump distribution: {multi_suit_df['trump'].value_counts().to_dict()}")

## 5.1 `plot_hand_value_by_trump_suit()` - hand value by suit

Box plots showing hand_value distribution for each trump suit. Includes variance annotations.

In [ ]:
from bid_euchre.diagnostics import plot_hand_value_by_trump_suit

fig = plot_hand_value_by_trump_suit(multi_suit_df, show_variance=True)
plt.show()

## 5.2 `plot_outcome_by_trump_suit()` - tricks won by suit

Outcome distribution showing if some suits systematically win more tricks.

In [ ]:
from bid_euchre.diagnostics import plot_outcome_by_trump_suit

fig = plot_outcome_by_trump_suit(multi_suit_outcome_df, outcome='tricks_won')
plt.show()

## 5.3 `plot_feature_heatmap_by_suit()` - feature means by suit

Heatmap showing which features vary most across trump suits.

In [ ]:
from bid_euchre.diagnostics import plot_feature_heatmap_by_suit

fig = plot_feature_heatmap_by_suit(multi_suit_df, normalize=True)
plt.show()

## 5.4 `plot_suit_variance_summary()` - variance comparison

Bar chart comparing variance of hand_value across suits.

In [ ]:
from bid_euchre.diagnostics import plot_suit_variance_summary

fig = plot_suit_variance_summary(multi_suit_df, column='feat_hand_value')
plt.show()

---
# Part 8: Distribution Analysis (CDF/CCDF)

CDF and CCDF plots for analyzing distribution shapes and tail behavior.

**CDF (Cumulative Distribution Function)**: Shows P(X ≤ x) — the probability a value is less than or equal to x.
- Useful for understanding distribution shape
- Quartile reference lines show median and spread

**CCDF (Complementary CDF)**: Shows P(X > x) — the probability a value exceeds x.
- Log scale reveals tail behavior
- Useful for identifying rare high-value hands

## 6.1 `plot_cdf()` - Cumulative Distribution Function

CDF for hand_value showing probability distribution shape with quartile markers.

In [ ]:
from bid_euchre.diagnostics import plot_cdf

# Basic CDF of hand_value
fig = plot_cdf(df, column='feat_hand_value')
plt.show()

In [ ]:
# CDF grouped by contract_type - compare distributions across categories
fig = plot_cdf(df, column='feat_hand_value', group_by='contract_type')
plt.show()

## 6.2 `plot_ccdf()` - Complementary CDF (Tail Analysis)

CCDF with log scale reveals tail behavior — useful for identifying rare high-value hands.

In [ ]:
from bid_euchre.diagnostics import plot_ccdf

# CCDF of hand_value with log scale - reveals tail distribution
fig = plot_ccdf(df, column='feat_hand_value', log_scale=True)
plt.show()

## 6.3 CDF of Tricks Won

CDF of simulation outcomes — shows probability of winning ≤ N tricks.

In [ ]:
# CDF of tricks_won - uses outcome_df from Part 3
fig = plot_cdf(outcome_df, column='tricks_won', group_by='contract_type')
plt.show()

## 6.4 CCDF of Tricks Won

CCDF shows P(tricks > N) — useful for analyzing win probability thresholds.

In [ ]:
# CCDF of tricks_won - P(tricks > N) by contract type
# At x=5, the CCDF shows win probability (≥6 tricks)
fig = plot_ccdf(outcome_df, column='tricks_won', log_scale=False, group_by='contract_type')
plt.show()

In [ ]:
# Cleanup temp directory
shutil.rmtree(output_dir)
print("Cleaned up temp directory")

# Quick Reference

## Diagnostic Charts (`bid_euchre.diagnostics`)

### Bidless-Specific Analysis (NEW)

| Chart | Purpose | Key Parameters |
|-------|---------|----------------|
| hand_value vs tricks_won | Predictive accuracy validation | Scatter + residual plot, R² |
| Feature stability | Cross-contract distribution comparison | Histograms + variance ratios |
| Contract-specific importance | Top features by contract type | Bar charts by correlation |
| Seat position effects | Dealer/leader/seat bias detection | Box plots + ANOVA tests |

### Feature Analysis Charts

| Function | Purpose | Key Parameters |
|----------|---------|----------------|
| `plot_hand_value_by_seat(df)` | Seat balance check | `df` with `seat`, `feat_hand_value` |
| `plot_hand_value_by_contract(df)` | Contract comparison | `df` with `contract_type`, `feat_hand_value` |
| `plot_feature_distributions(df)` | Feature histograms | `features=None` for top 9 by variance |
| `plot_feature_correlation(df)` | Correlation heatmap | `features=None` for top 10 by variance |
| `plot_rolling_mean(df, column)` | Drift detection | `window=100` default |
| `plot_feature_vs_label(df, feature)` | Scatter + boxplot | `label='feat_hand_value'` default |

### Outcome Evaluation Charts

| Function | Purpose | Key Parameters |
|----------|---------|----------------|
| `plot_feature_vs_outcome(df, feature)` | Feature vs outcome with correlation | `outcome='tricks_won'` |
| `plot_outcome_distributions(df, outcome)` | Outcome by category | `group_by='contract_type'` |
| `plot_feature_outcome_correlation(df)` | Feature importance bar chart | `outcome='tricks_won'`, `top_n=15` |

### Trump Suit Analysis Charts

| Function | Purpose | Key Parameters |
|----------|---------|----------------|
| `plot_hand_value_by_trump_suit(df)` | Hand value by suit | `show_variance=True` |
| `plot_outcome_by_trump_suit(df)` | Tricks won by suit | `outcome='tricks_won'` |
| `plot_feature_heatmap_by_suit(df)` | Feature means by suit | `normalize=True`, `features=None` |
| `plot_suit_variance_summary(df)` | Variance comparison | `column='feat_hand_value'` |

### Strategy Comparison Charts

| Function | Purpose | Key Parameters |
|----------|---------|----------------|
| `plot_win_rate_heatmap(matchup_results)` | All-vs-all win rate matrix | `metric='win_rate'` |
| `plot_tricks_distribution_comparison(matchup_results)` | Violin plots by matchup | `team=0` |
| `plot_strategy_delta_bars(baseline, comparisons)` | Delta vs baseline | `metric='mean_tricks'` |
| `plot_self_play_control(self_play_results)` | Self-play sanity check | `expected_mean=5.0` |

### Distribution Analysis Charts

| Function | Purpose | Key Parameters |
|----------|---------|----------------|
| `plot_cdf(df, column)` | Cumulative distribution | `group_by=None` for overlaid CDFs |
| `plot_ccdf(df, column)` | Tail distribution (1-CDF) | `log_scale=True` for heavy tails |

## Evaluation Charts (`bid_euchre.reporting.validation`)

| Function | Purpose | Input Format |
|----------|---------|---------------|
| `plot_feature_distributions(fbc, dir)` | By-contract histograms | `Dict[str, List[Dict]]` |
| `plot_feature_correlation(features, dir)` | Correlation matrix | `List[Dict]` |
| `plot_hand_value_by_contract(fbc, dir)` | Contract box plots | `Dict[str, List[Dict]]` |
| `generate_validation_plots(fbc, dir)` | All of the above | `Dict[str, List[Dict]]` |